In [ ]:
# import  os
# num_cores = "1"
# os.environ["OPENBLAS_NUM_THREADS"] = num_cores
# os.environ["OMP_NUM_THREADS"] = num_cores
# os.environ["MKL_NUM_THREADS"] = num_cores

In [1]:
import sys
sys.path.append('../')

import numpy as np
import scipy.sparse as ssp
import matplotlib.pyplot as plt
import qutip as qt
import scqubits as scq
from matplotlib.colors import LogNorm
from tqdm import tqdm
from qutip.qip.operations import rz, cz_gate
import cmath
from sympy import symbols
from scipy.sparse.linalg import eigsh
import utils_2Q_gate_zp as ut
from joblib import Parallel, delayed
from IPython.display import display, Math
# ut.set_fig_font() ### Set various sizes in plotting
import networkx as nx
from multiprocessing import Pool
import pandas as pd

In [2]:
truc1, truc_tot, charge_pick  = 300, 1000, True
truc_tot_2 = 400
folder = f'../../data/3ncut_two_zeropi/truc1={truc1}_truc2={truc_tot}_pick={charge_pick}/'
eval_tot = pd.read_csv(folder+ 'eval_tot.txt').to_numpy().flatten()
n_theta0_dress = 2*np.pi* np.load(folder+'n_theta0_dress.npy')
n_theta1_dress = 2*np.pi* np.load(folder+'n_theta1_dress.npy')
hspace_full = pd.read_csv(folder+ 'hspace_full.txt').to_numpy().flatten().tolist()

truc_list = np.arange(truc_tot_2)
hspace_full = hspace_full[:truc_tot_2]
eval_tot = eval_tot[:truc_tot_2]
n_theta0_dress = ut.truncate_2(n_theta0_dress, truc_list)
n_theta1_dress = ut.truncate_2(n_theta1_dress, truc_list)
logi_state = ['0-0', '0-2', '2-0', '2-2']

In [3]:
mid_state = '8-2'
idx_0 = hspace_full.index('0-2')
idx_1 = hspace_full.index('2-2')
drive_term = n_theta0_dress

idx_2 = hspace_full.index(mid_state)
core_states = logi_state + [mid_state]
W_0_2 = eval_tot[idx_2] - eval_tot[idx_0]
W_1_2 = eval_tot[idx_2] - eval_tot[idx_1]
print('E_45: W_0_2 = ', np.round(2*np.pi*W_0_2, 3), ', W_1_2 = ', np.round(2*np.pi*W_1_2, 3))

E_45: W_0_2 =  45.546 , W_1_2 =  23.753


### Truncation Estimate

In [4]:
max_n_ij = np.max(np.abs(drive_term.full()))
A = 0.02
population_rate = np.zeros((truc_tot, truc_tot), dtype=np.complex128)
population_rate_log = np.zeros((truc_tot, truc_tot), dtype=np.complex128)
rabi_df = []
G = nx.DiGraph()
for i, s_i in enumerate(hspace_full):
    for j, s_j in enumerate(hspace_full):
        if i < j:
            n_ij = np.abs(drive_term[i, j])/max_n_ij
            delta_1 = abs(W_0_2 - (eval_tot[j] - eval_tot[i]))
            delta_2 = abs(W_1_2 - (eval_tot[j] - eval_tot[i]))

            population_rate[i,j] = ((A*n_ij)**2) / ((A*n_ij)**2 + delta_1**2) + ((A*n_ij)**2) / ((A*n_ij)**2 + delta_2**2)
            if population_rate[i,j] > 0:
                population_rate_log[i, j] = -np.log(population_rate[i,j])
                G.add_edge(s_i, s_j, weight=population_rate_log[i, j]) # Construct the graph

    rabi_df.append({"order": i, "i": s_i})
rabi_df = pd.DataFrame(rabi_df)
rabi_df.index = rabi_df["i"]
print('shape(population_rate_log)=', np.shape(population_rate_log))
# rabi_df

shape(population_rate_log)= (1000, 1000)


In [5]:
def shortest_path_to_core(target):
    shortest_path = ""
    shortest_path_len = np.inf
    for source in core_states[:-1]:
        if nx.has_path(G, source, target):
            path = nx.shortest_path(G, source=source, target=target,
                                    weight="weight")
            path_len = nx.shortest_path_length(G, source=source, target=target,
                                                weight="weight")
        if path_len < shortest_path_len:
            shortest_path_len = path_len
            shortest_path = ",".join([str(x) for x in path])
    return target, (shortest_path_len, shortest_path)

cutoff = 2
def all_path_to_core(target):
    path_tot = []
    if target in core_states:
        weight_tot = 1
    else:
        weight_tot = 0
        for source in core_states:
            for path in nx.all_simple_paths(G, source, target, cutoff=cutoff):
                weight_tot += np.exp( - nx.path_weight(G, path,'weight') )
                path_tot.append(path)
    return target, (weight_tot, path_tot)


### All path

In [6]:
# Find all_path_to_core
pool = Pool(processes=150)
shortest_path = {}
rabi_df["path"] = ""
rabi_df["path_len"] = 1.0
for idx, path in tqdm(pool.imap_unordered(all_path_to_core, hspace_full),
                total=truc_tot):
    shortest_path[idx[0]] = path
    rabi_df.at[idx, "path"] = path[1]
    rabi_df.at[idx, "path_len"] = path[0]

df_all = rabi_df.sort_values("path_len", ascending=False)
print('all_path -- cutoff=', cutoff)
df_all.iloc[:30]

 40%|████      | 400/1000 [00:03<00:05, 109.79it/s]


all_path -- cutoff= 2


,order,i,path,path_len
i,,,,
0-0,0,0-0,[],1.000000e+00+0.000000e+ 00j
2-0,4,2-0,[],1.000000e+00+0.000000e+ 00j
2-2,13,2-2,[],1.000000e+00+0.000000e+ 00j
8-2,37,8-2,[],1.000000e+00+0.000000e+ 00j
0-2,3,0-2,[],1.000000e+00+0.000000e+ 00j
8-0,14,8-0,"[[0-0, 0-1, 8-0], [0-0, 1-0, 8-0], [0-0, 0-2, ...",8.235059e-05+0.000000e+ 00j
12-2,64,12-2,"[[0-0, 0-1, 12-2], [0-0, 1-0, 12-2], [0-0, 0-2...",7.004084e-05+0.000000e+ 00j
4-9,63,4-9,"[[0-0, 0-1, 4-9], [0-0, 1-0, 4-9], [0-0, 0-2, ...",3.262049e-05+0.000000e+ 00j
1-2,11,1-2,"[[0-0, 0-1, 1-2], [0-0, 1-0, 1-2], [0-0, 0-2, ...",2.522371e-05+0.000000e+ 00j


In [7]:
print(f' {mid_state}_all_{truc_tot_2} :')
data = df_all['i'].to_numpy().tolist()
dim = 10
for i in range(0, len(data), dim):  # Step size of 10
    if i%50==0:
        print('')
    print(", ".join(f"'{x}'" for x in data[i:i + dim]), ',')

 8-2_all_400 :

'0-0', '2-0', '2-2', '8-2', '0-2', '8-0', '12-2', '4-9', '1-2', '20-5' ,
'1-0', '5-2', '1-4', '5-0', '30-2', '22-2', '8-5', '34-2', '5-8', '2-1' ,
'9-2', '13-2', '5-5', '0-1', '2-5', '0-5', '4-5', '15-5', '9-5', '15-0' ,
'15-2', '24-2', '13-9', '26-5', '25-2', '9-1', '8-1', '25-4', '4-1', '5-4' ,
'37-2', '18-5', '9-4', '26-2', '9-9', '20-2', '4-13', '18-2', '5-16', '20-0' ,

'9-12', '18-0', '12-9', '18-1', '5-9', '0-21', '4-12', '12-4', '20-1', '1-18' ,
'1-13', '15-1', '18-8', '1-9', '28-2', '0-8', '8-18', '8-9', '2-8', '2-12' ,
'44-2', '12-5', '1-21', '9-13', '4-8', '24-9', '26-8', '12-1', '8-4', '12-0' ,
'22-4', '2-30', '13-4', '2-25', '26-0', '15-8', '8-8', '46-2', '22-9', '13-0' ,
'2-36', '0-30', '13-5', '1-20', '41-1', '4-2', '25-0', '35-2', '30-5', '1-8' ,

'1-1', '45-0', '12-12', '45-2', '35-0', '4-4', '0-18', '2-35', '9-8', '54-2' ,
'0-25', '30-4', '13-1', '33-5', '51-0', '9-0', '33-1', '30-0', '5-18', '0-45' ,
'0-35', '51-2', '1-42', '4-24', '12-8', '0-36', '56

In [8]:
# num_state = 200
# df_all_truc = df_all.iloc[:num_state].sort_values("order", ascending=True)
# print(f'state_all ({num_state}/{truc_tot_2}) :')
# data = df_all_truc['i']
# dim = 10
# for i in range(0, len(data), dim):  # Step size of 10
#     print(", ".join(f"'{x}'" for x in data[i:i + dim]), ',')

# index_all = [hspace_full.index(i) for i in df_all_truc['i']]
# print(index_all)

### Shortest path

In [9]:
# Find shortest path for each node
pool = Pool(processes=150)
shortest_path = {}
rabi_df["path"] = ""
rabi_df["path_len"] = 1.0
for idx, path in tqdm(pool.imap_unordered(shortest_path_to_core, hspace_full),
                total=truc_tot):
    shortest_path[idx[0]] = path
    rabi_df.at[idx, "path"] = path[1]
    rabi_df.at[idx, "path_len"] = np.exp(-path[0])

df_short = rabi_df.sort_values("path_len", ascending=False)
print('shortest_path:')
df_short.iloc[:30]

 40%|████      | 400/1000 [00:00<00:01, 459.55it/s]

shortest_path:


,order,i,path,path_len
i,,,,
8-2,37,8-2,"2-2,8-2",1.000000e+00-0.000000e+ 00j
0-0,0,0-0,0-0,1.000000e+00+0.000000e+ 00j
0-2,3,0-2,0-2,1.000000e+00+0.000000e+ 00j
2-0,4,2-0,2-0,1.000000e+00+0.000000e+ 00j
2-2,13,2-2,2-2,1.000000e+00+0.000000e+ 00j
8-0,14,8-0,"2-0,8-0",5.935617e-05-0.000000e+ 00j
1-2,11,1-2,"0-2,1-2",2.522370e-05-0.000000e+ 00j
1-0,2,1-0,"0-0,1-0",2.412400e-05-0.000000e+ 00j
12-2,64,12-2,"2-2,8-2,12-2",2.334676e-05-0.000000e+ 00j


In [10]:
print(f' {mid_state}_short_{truc_tot_2} :')
data = df_short['i'].to_numpy().tolist()
dim = 10
for i in range(0, len(data), dim):  # Step size of 10
    if i%50==0:
        print('')
    print(", ".join(f"'{x}'" for x in data[i:i + dim]), ',')

 8-2_short_400 :

'8-2', '0-0', '0-2', '2-0', '2-2', '8-0', '1-2', '1-0', '12-2', '5-2' ,
'5-0', '1-4', '4-9', '20-5', '30-2', '22-2', '8-5', '2-1', '34-2', '0-1' ,
'2-5', '5-8', '9-2', '13-2', '0-5', '5-5', '9-5', '4-5', '15-0', '15-2' ,
'15-5', '9-1', '24-2', '8-1', '5-4', '13-9', '26-5', '25-2', '4-1', '25-4' ,
'26-2', '37-2', '20-2', '18-2', '18-5', '9-4', '20-0', '18-0', '9-9', '5-9' ,

'0-21', '1-13', '4-13', '5-16', '1-9', '0-8', '9-12', '8-9', '12-9', '2-8' ,
'18-1', '4-12', '20-1', '1-18', '12-4', '15-1', '12-5', '18-8', '4-8', '12-1' ,
'28-2', '8-4', '8-18', '26-0', '2-12', '44-2', '1-21', '13-0', '12-0', '1-20' ,
'9-13', '24-9', '4-2', '13-5', '26-8', '30-5', '22-4', '1-1', '2-30', '45-0' ,
'2-25', '13-4', '15-8', '45-2', '9-8', '13-1', '1-8', '0-18', '0-45', '46-2' ,

'25-0', '22-9', '8-8', '2-36', '35-2', '35-0', '0-30', '41-0', '41-1', '39-0' ,
'9-0', '35-5', '2-21', '56-2', '4-0', '39-2', '41-2', '12-12', '5-1', '37-5' ,
'4-4', '2-35', '33-0', '33-2', '54-2', '46-0', '0-

In [11]:
# num_state = 200
# df_short_truc = df_short.iloc[:num_state].sort_values("order", ascending=True)
# print('state_short :')
# data = df_short_truc['i']
# for i in range(0, len(data), dim):  # Step size of 10
#     print(", ".join(f"'{x}'" for x in data[i:i + dim]), ',')

# index_all = [hspace_full.index(i) for i in df_short_truc['i']]
# print(index_all)

In [12]:
=

SyntaxError: invalid syntax (1763773627.py, line 1)

In [14]:
short_82_v0 = [
'0-0', '0-2', '2-0', '8-2', '2-2', '12-2', '4-9', '1-2', '1-0', '5-2' ,
'5-0', '8-5', '22-2', '9-2', '2-1', '5-8', '5-5', '0-1', '34-2', '2-5' ,
'13-2', '4-2', '8-0', '1-1', '0-5', '8-9', '15-2', '26-2', '30-2', '1-4' ,
'9-0', '20-2', '20-5', '12-5', '4-0', '15-5', '5-1', '18-2', '12-0', '24-2' ,
'26-5', '1-5', '15-0', '13-0', '25-2', '35-2', '8-1', '33-2', '9-5', '4-5' ,

'12-8', '9-8', '1-8', '37-2', '5-12', '25-0', '9-4', '1-25', '22-5', '56-2' ,
'18-4', '13-9', '4-4', '2-4', '15-4', '50-2', '13-5', '9-9', '5-9', '4-1' ,
'20-0', '25-5', '15-9', '18-5', '9-1', '30-5', '2-21', '26-9', '1-13', '20-9' ,
'45-2', '18-0', '5-16', '22-0', '34-5', '15-1', '5-4', '0-21', '20-4', '26-0' ,
'25-4', '25-1', '0-4', '9-12', '24-5', '37-5', '24-0', '1-18', '12-9', '44-2' ,

'25-8', '2-12', '41-2', '4-16', '4-12', '4-21', '8-12', '18-9', '18-1', '45-0' ,
'24-9', '26-8', '39-2', '4-8', '8-4', '34-0', '22-8', '35-4', '12-4', '30-0' ,
'12-1', '28-2', '18-8', '0-8', '35-5', '35-0', '9-16', '37-0', '2-8', '1-9' ,
'13-8', '46-2', '2-26', '46-0', '2-9', '41-0', '4-13', '39-0', '2-25', '0-12' ,
'22-9', '33-0', '33-4', '22-4', '54-2', '8-18', '28-1', '0-45', '50-0', '24-1' ,

'28-5', '30-1', '15-8', '34-1', '22-1', '20-1', '13-16', '12-18', '0-9', '24-8' ,
'13-1', '51-2', '33-5', '2-30', '1-21', '4-18', '59-0', '15-12', '0-25', '8-24' ,
'8-8', '9-18', '44-0', '26-1', '65-0', '56-0', '41-1', '41-4', '5-30', '39-4' ,
'30-4', '58-0', '5-13', '18-12', '0-30', '45-4', '0-39', '13-4', '46-1', '28-0' ,
'2-18', '0-24', '0-13', '9-13', '30-8', '69-0', '41-5', '33-1', '37-8', '4-24' ,
]
short_82 = [
'0-0', '2-0', '2-2', '8-2', '0-2', '8-0', '12-2', '4-9', '1-2', '20-5' ,
'1-0', '5-2', '1-4', '5-0', '30-2', '22-2', '8-5', '34-2', '5-8', '2-1' ,
'9-2', '13-2', '5-5', '0-1', '2-5', '0-5', '4-5', '15-5', '9-5', '15-0' ,
'15-2', '24-2', '13-9', '26-5', '25-2', '9-1', '8-1', '25-4', '4-1', '5-4' ,
'37-2', '18-5', '9-4', '26-2', '9-9', '20-2', '4-13', '18-2', '5-16', '20-0' ,

'9-12', '18-0', '12-9', '18-1', '5-9', '0-21', '4-12', '12-4', '20-1', '1-18' ,
'1-13', '15-1', '18-8', '1-9', '28-2', '0-8', '8-18', '8-9', '2-8', '2-12' ,
'44-2', '12-5', '1-21', '9-13', '4-8', '24-9', '26-8', '12-1', '8-4', '12-0' ,
'22-4', '2-30', '13-4', '2-25', '26-0', '15-8', '8-8', '46-2', '22-9', '13-0' ,
'2-36', '0-30', '13-5', '1-20', '41-1', '4-2', '25-0', '35-2', '30-5', '1-8' ,

'1-1', '45-0', '12-12', '45-2', '35-0', '4-4', '0-18', '2-35', '9-8', '54-2' ,
'0-25', '30-4', '13-1', '33-5', '51-0', '9-0', '33-1', '30-0', '5-18', '0-45' ,
'0-35', '51-2', '1-42', '4-24', '12-8', '0-36', '56-2', '33-2', '41-0', '39-0' ,
'5-1', '35-5', '22-0', '4-0', '44-0', '13-12', '50-2', '2-9', '39-2', '2-21' ,
'46-0', '35-1', '41-2', '37-5', '26-1', '34-5', '33-0', '34-0', '24-4', '0-24' ,

'1-39', '50-0', '2-13', '25-5', '1-25', '20-8', '1-5', '12-13', '4-21', '35-4' ,
'12-20', '5-21', '58-1', '39-5', '41-5', '20-9', '4-30', '22-5', '18-4', '59-0' ,
'18-9', '56-0', '5-12', '0-20', '4-25', '28-1', '65-0', '34-4', '4-20', '28-5' ,
'39-4', '25-8', '24-5', '1-16', '24-0', '20-4', '46-4', '1-57', '25-1', '34-1' ,
'26-9', '1-12', '69-0', '15-9', '2-4', '0-16', '24-1', '30-9', '41-4', '15-4' ,
]

list1 = short_82_v0
list2 = short_82
common_elements = [item for item in list1 if item in list2]
only_in_list1 = [item for item in list1 if item not in list2]
only_in_list2 = [item for item in list2 if item not in list1]
unique_elements = only_in_list1 + only_in_list2

print(f" Only in 1:", len(only_in_list1), only_in_list1)
print('index only in 1:', [list1.index(i) for i in only_in_list1])
print("Only in 2:", len(only_in_list2), only_in_list2)
print('index only in 2:', [list2.index(i) for i in only_in_list2])

 Only in 1: 32 ['0-4', '4-16', '8-12', '22-8', '9-16', '37-0', '13-8', '2-26', '0-12', '33-4', '30-1', '22-1', '13-16', '12-18', '0-9', '24-8', '4-18', '15-12', '8-24', '9-18', '5-30', '58-0', '5-13', '18-12', '45-4', '0-39', '46-1', '28-0', '2-18', '0-13', '30-8', '37-8']
index only in 1: [92, 103, 106, 116, 126, 127, 130, 132, 139, 142, 151, 154, 156, 157, 158, 159, 165, 167, 169, 171, 178, 181, 182, 183, 185, 186, 188, 189, 190, 192, 194, 198]
Only in 2: 32 ['2-36', '1-20', '12-12', '0-18', '2-35', '51-0', '5-18', '0-35', '1-42', '0-36', '13-12', '35-1', '24-4', '1-39', '2-13', '20-8', '12-13', '12-20', '5-21', '58-1', '39-5', '4-30', '0-20', '4-25', '34-4', '4-20', '1-16', '46-4', '1-57', '1-12', '0-16', '30-9']
index only in 2: [90, 93, 102, 106, 107, 114, 118, 120, 122, 125, 135, 141, 148, 150, 152, 155, 157, 160, 161, 162, 163, 166, 173, 174, 177, 178, 183, 186, 187, 191, 195, 197]
